# Benchmark Scenario 005 — So sánh số bit (16 / 32 / 64)

**KB5**: Cố định framework CSQ + backbone ViT-B_32 trên CIFAR-10, thay đổi số bit (16, 32, 64) để trả lời câu hỏi *"MAP có tăng đều khi tăng số bit không?"*

## Workflow
1. Chạy **Cell 1–4** (mount Drive → clone repo → cài deps → symlink) mỗi khi mở Colab runtime mới.
2. Chạy **Cell 5** (config) — mặc định `EPOCH=150`, `TEST_MAP=30` cho Colab.
3. Chạy **Cell 6** (smoke test) — 3 bit × epoch 10 × test_map 5 tuần tự, xác nhận pipeline không crash.
4. Mở **3 runtime Colab song song**, mỗi runtime chạy 1 trong Cell 7–9 (bit 16 / 32 / 64) để rút ngắn wall-clock.

Kết quả ghi vào `Checkpoints_Results/CSQ-ViT-B_32-cifar10-Bit{bit}/` (persist qua Drive symlink).

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
# Cell 2 — Clone repo và chuyển vào thư mục dự án
!git clone https://github.com/hatuan314/VisionTransformerHashing.git
%cd VisionTransformerHashing

In [ ]:
# Cell 3 — Cài dependency thiếu trên Colab
!pip install ml_collections

In [ ]:
# Cell 4 — Tạo symlink tới Drive (pretrainedVIT, dataset, Checkpoints_Results)
import os

DRIVE_BASE = '/content/gdrive/MyDrive/master_is/semester_3/IR/VTS-LAB'

for link in ['pretrainedVIT', 'dataset', 'Checkpoints_Results']:
    if os.path.islink(link):
        os.remove(link)

os.symlink(f'{DRIVE_BASE}/pretrainedVIT', 'pretrainedVIT')
os.symlink(f'{DRIVE_BASE}/dataset', 'dataset')

os.makedirs(f'{DRIVE_BASE}/Checkpoints_Results', exist_ok=True)
os.symlink(f'{DRIVE_BASE}/Checkpoints_Results', 'Checkpoints_Results')

!ls pretrainedVIT/
!ls dataset/
!ls Checkpoints_Results/

In [ ]:
# Cell 5 — Config
EPOCH = 150        # smoke: Cell 6 dùng SMOKE_EPOCH=10 riêng
TEST_MAP = 30
FRAMEWORK = "CSQ"
DATASET = "cifar10"
BACKBONE = "ViT-B_32"
BITS = [16, 32, 64]

def save_path_for(bit):
    return f"Checkpoints_Results/{FRAMEWORK}-{BACKBONE}-{DATASET}-Bit{bit}"

print(f"EPOCH={EPOCH} | TEST_MAP={TEST_MAP} | DATASET={DATASET} | FRAMEWORK={FRAMEWORK} | BACKBONE={BACKBONE}")
print("Bits:", BITS)
for bit in BITS:
    print(f"  Bit{bit:2d} -> {save_path_for(bit)}")

In [ ]:
# Cell 6 — Smoke test: 3 bit × epoch 10 tuần tự
import subprocess

SMOKE_EPOCH, SMOKE_TEST_MAP = 10, 5
results = {}
for bit in BITS:
    cmd = [
        "python", f"{FRAMEWORK}.py",
        "--dataset", DATASET,
        "--bit", str(bit),
        "--epoch", str(SMOKE_EPOCH),
        "--test_map", str(SMOKE_TEST_MAP),
        "--backbone", BACKBONE,
        "--save_path", save_path_for(bit),
    ]
    rc = subprocess.call(cmd)
    results[bit] = "OK" if rc == 0 else f"FAIL(rc={rc})"

print("\n=== Smoke Test Results ===")
for bit, st in results.items():
    print(f"  Bit{bit:2d}  {st}")

In [ ]:
# Cell 7 — Train bit=16 (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset {DATASET} --bit 16 --epoch {EPOCH} --test_map {TEST_MAP} --backbone {BACKBONE} --save_path {save_path_for(16)}

In [ ]:
# Cell 8 — Train bit=32 (chạy trên 1 Colab runtime riêng)
!python CSQ.py --dataset {DATASET} --bit 32 --epoch {EPOCH} --test_map {TEST_MAP} --backbone {BACKBONE} --save_path {save_path_for(32)}

In [ ]:
# Cell 9 — Train bit=64 (chạy trên 1 Colab runtime riêng)
# Lưu ý: nếu OOM với batch_size=32, giảm batch_size trong get_config() rồi rerun
!python CSQ.py --dataset {DATASET} --bit 64 --epoch {EPOCH} --test_map {TEST_MAP} --backbone {BACKBONE} --save_path {save_path_for(64)}